# Cell 1: Imports and Setup

In [ ]:
!pip install -U google-generativeai

In [ ]:
import time
from typing import Optional, Dict, List, Any
import requests
import http.client
import json
import google.generativeai as genai
from IPython.display import display, Markdown

API_KEY_COINGECKO = '$$$$$$$$$'
API_KEY_SERPER = '$$$$$$'
API_KEY_GOOGLE_AI = '$$$$$$$'



genai.configure(api_key=API_KEY_GOOGLE_AI)

# Cell 2: CoinGecko & Helper Functions

In [ ]:
def get_json_data(url: str) -> dict:
    headers = {'x-cg-demo-api-key': API_KEY_COINGECKO}
    response = requests.get(url, headers=headers)
    if response.status_code == 200:
        return response.json()
    else:
        raise Exception(f"Error fetching data: {response.status_code}")

def rate_limit(response: dict) -> bool:
    if "status" in response:
        if response["status"].get("error_code") == 429:
            print(f"Rate limit error: {response['status']['error_message']}")
            return True
    return False

def get_coin_by_id(id: str) -> dict:
    data = get_json_data(f"https://api.coingecko.com/api/v3/coins/{id}")
    if rate_limit(data):
        raise Exception("Rate limit reached")
    return data

def get_coins_markets(currency: str = "usd", order: str = "market_cap_desc", per_page: int = 250, page: int = 1) -> List[dict]:
    data = get_json_data(
        f"https://api.coingecko.com/api/v3/coins/markets?vs_currency={currency}&order={order}&per_page={per_page}&page={page}&sparkline=false"
    )
    if rate_limit(data):
        raise Exception("Rate limit reached")
    return data

def get_detailed_coin_info(ticker: str) -> Optional[Dict]:
    try:
        coins_list = get_coins_markets()
        coin_data = next((coin for coin in coins_list if coin['symbol'].upper() == ticker.upper()), None)

        if coin_data:
            coin_id = coin_data['id']
            coin_dict = get_coin_by_id(coin_id)

            if coin_dict:
                return {
                    "id": coin_id,
                    "symbol": coin_dict["symbol"],
                    "name": coin_dict["name"],
                    "price": coin_dict["market_data"]["current_price"].get("usd", 0),
                    "market_cap_rank": coin_dict["market_cap_rank"],
                    "description": coin_dict["description"].get("en", ""),
                }
    except Exception as e:
        print(f"Error in API call for {ticker}: {e}")
    return None

def get_top_coins(n: int = 100) -> List[Dict]:
    coins = get_coins_markets(per_page=n)
    return [{
        "rank": coin["market_cap_rank"],
        "symbol": coin["symbol"],
        "name": coin["name"],
        "price": coin["current_price"],
        "market_cap": coin["market_cap"],
        "volume_24h": coin["total_volume"],
        "change_24h": coin["price_change_percentage_24h"],
    } for coin in coins]


Note: These helper functions are used to interact with the CoinGecko API.
They handle JSON data retrieval, specific coin data fetching, and rate limiting.

# Search Tool

In [ ]:
def search_coin(coin_name: str) -> Dict[str, Any]:
    conn = http.client.HTTPSConnection("google.serper.dev")
    payload = json.dumps({
        "q": f"{coin_name} cryptocurrency analysis"
    })
    headers = {
        'X-API-KEY': API_KEY_SERPER,
        'Content-Type': 'application/json'
    }
    conn.request("POST", "/search", payload, headers)
    res = conn.getresponse()
    data = res.read()
    return json.loads(data.decode("utf-8"))


#Using Gemini Flash

In [ ]:
def analyze_coin(coin_data: Dict[str, Any], search_results: Dict[str, Any]) -> str:
    model = genai.GenerativeModel("gemini-1.5-flash")

    prompt = f"""Analyze this cryptocurrency data and search results. Provide a general report, recent trends, and potential alpha:

Coin Data:
{json.dumps(coin_data, indent=2)}

Search Results:
{json.dumps(search_results, indent=2)}"""

    response = model.generate_content(prompt)
    return response.text

#Agent Workflow

In [ ]:
def crypto_analysis(coin_name: str) -> Dict[str, Any]:
    # Step 1: Get detailed coin info from CoinGecko
    time.sleep(6)  # Rate limiting
    coin_data = get_detailed_coin_info(coin_name)
    if not coin_data:
        return {"error": f"Unable to fetch data for {coin_name}"}

    # Step 2: Perform web search for recent news and analysis
    search_results = search_coin(coin_data['name'])

    # Step 3: Get top 10 coins for market context
    top_coins = get_top_coins(10)

    # Step 4: Analyze data with Google AI
    analysis = analyze_coin(coin_data, search_results)

    # Step 5: Compile all information into a comprehensive report
    report = {
        "coin_data": coin_data,
        "market_context": {
            "top_10_coins": top_coins,
            "coin_rank": next((coin['rank'] for coin in top_coins if coin['symbol'] == coin_name.lower()), "Not in top 10")
        },
        "recent_news": search_results,
        "ai_analysis": analysis
    }

    return report

# Report Generation


---



In [ ]:
def generate_markdown_report(report: Dict[str, Any]) -> str:
    coin_data = report.get('coin_data', {})
    market_context = report.get('market_context', {})

    md_content = f"""
# Cryptocurrency Analysis Report for {coin_data.get('name', 'Unknown')}

## Coin Data
- Name: {coin_data.get('name', 'Unknown')}
- Symbol: {coin_data.get('symbol', 'Unknown')}
- Current Price: ${coin_data.get('price', 'N/A'):,.2f}
- Market Cap Rank: {coin_data.get('market_cap_rank', 'N/A')}


## Description
{coin_data.get('description', 'No description available.')}

## Market Context
### Top 10 Coins:
{chr(10).join([f"{i+1}. {coin.get('name', 'Unknown')} ({coin.get('symbol', 'Unknown')})" for i, coin in enumerate(market_context.get('top_10_coins', []))])}

## AI Analysis
{report.get('ai_analysis', 'No AI analysis available')}
    """
    return md_content

# Main function to run the analysis


In [ ]:
def main():
    coin_name = input("Enter the name of the cryptocurrency to analyze: ")
    print(f"Analyzing {coin_name}...")

    try:
        analysis_report = crypto_analysis(coin_name)
        if "error" in analysis_report:
            print(f"Error: {analysis_report['error']}")
        else:
            md_report = generate_markdown_report(analysis_report)

            # Display in Jupyter Notebook
            display(Markdown(md_report))

            # Save to file
            with open(f"{coin_name}_analysis.md", "w") as f:
                f.write(md_report)
            print(f"Analysis report saved as {coin_name}_analysis.md")
    except Exception as e:

        print(f"An error occurred: {e}")

# Cell 8: Run the main function
if __name__ == "__main__":
    main()

Enter the name of the cryptocurrency to analyze: BTC
Analyzing BTC...



# Cryptocurrency Analysis Report for Bitcoin

## Coin Data
- Name: Bitcoin
- Symbol: btc
- Current Price: $55,976.00
- Market Cap Rank: 1


## Description
Bitcoin is the first successful internet money based on peer-to-peer technology; whereby no central bank or authority is involved in the transaction and production of the Bitcoin currency. It was created by an anonymous individual/group under the name, Satoshi Nakamoto. The source code is available publicly as an open source project, anybody can look at it and be part of the developmental process.

Bitcoin is changing the way we see money as we speak. The idea was to produce a means of exchange, independent of any central authority, that could be transferred electronically in a secure, verifiable and immutable way. It is a decentralized peer-to-peer internet currency making mobile payment easy, very low transaction fees, protects your identity, and it works anywhere all the time with no central authority and banks.

Bitcoin is designed to have only 21 million BTC ever created, thus making it a deflationary currency. Bitcoin uses the <a href="https://www.coingecko.com/en?hashing_algorithm=SHA-256">SHA-256</a> hashing algorithm with an average transaction confirmation time of 10 minutes. Miners today are mining Bitcoin using ASIC chip dedicated to only mining Bitcoin, and the hash rate has shot up to peta hashes.

Being the first successful online cryptography currency, Bitcoin has inspired other alternative currencies such as <a href="https://www.coingecko.com/en/coins/litecoin">Litecoin</a>, <a href="https://www.coingecko.com/en/coins/peercoin">Peercoin</a>, <a href="https://www.coingecko.com/en/coins/primecoin">Primecoin</a>, and so on.

The cryptocurrency then took off with the innovation of the turing-complete smart contract by <a href="https://www.coingecko.com/en/coins/ethereum">Ethereum</a> which led to the development of other amazing projects such as <a href="https://www.coingecko.com/en/coins/eos">EOS</a>, <a href="https://www.coingecko.com/en/coins/tron">Tron</a>, and even crypto-collectibles such as <a href="https://www.coingecko.com/buzz/ethereum-still-king-dapps-cryptokitties-need-1-billion-on-eos">CryptoKitties</a>.

## Market Context
### Top 10 Coins:
1. Bitcoin (btc)
2. Ethereum (eth)
3. Tether (usdt)
4. BNB (bnb)
5. Solana (sol)
6. USDC (usdc)
7. XRP (xrp)
8. Lido Staked Ether (steth)
9. Dogecoin (doge)
10. TRON (trx)

## AI Analysis
## Bitcoin Analysis Report: 

**General Report:**

Bitcoin, the first and most well-known cryptocurrency, is currently experiencing a downward trend. While still the dominant player in the cryptocurrency market, it is facing increasing competition from other cryptocurrencies with innovative features like smart contracts. 

**Recent Trends:**

* **Price decline:** Bitcoin is currently down about 3% and may fall further below the important support level of 56,561. 
* **Technical Analysis:** The overall sentiment is uncertain. Some analyses suggest a potential drop towards the 50,000 mark, while others indicate a medium-long term horizontal trend channel, suggesting investors are waiting for further signals.
* **ETF Outflows:** Outflows from Bitcoin ETFs contribute to the downward pressure.
* **Regulatory Uncertainty:** The potential sale of Bitcoin seized by the US government is raising concerns.
* **Risk-Off Sentiment:** A general risk-off sentiment in the market is also affecting Bitcoin's price.

**Potential Alpha:**

While there are currently bearish signals for Bitcoin, identifying potential alpha requires a nuanced approach:

* **Short-Term:** The current market trend suggests short-selling opportunities as Bitcoin may continue to decline. However, this strategy involves higher risk due to Bitcoin's volatility.
* **Long-Term:** Investing in Bitcoin is a long-term play, based on the belief that it will eventually become a dominant force in the global financial system. However, this strategy requires patience and a strong conviction in Bitcoin's long-term potential.
* **Alternative Cryptocurrencies:** The success of Bitcoin has spawned a plethora of other cryptocurrencies, some with innovative features that could surpass Bitcoin in the future. Investing in these alternatives might provide alpha in the long run.

**Key Considerations:**

* **Market Volatility:** The cryptocurrency market is highly volatile. Investments should be carefully managed with appropriate risk tolerance.
* **Regulatory Landscape:** The regulatory landscape surrounding cryptocurrencies is constantly evolving. This uncertainty can impact the price of Bitcoin and other cryptocurrencies.
* **Technology Development:** The ongoing development of blockchain technology and emerging cryptocurrencies can significantly impact Bitcoin's dominance in the future.

**Recommendation:**

* **Diversify:** Invest in a diversified portfolio of cryptocurrencies and traditional assets to mitigate risk.
* **Do your own research:** Thoroughly research and understand the fundamentals of Bitcoin and other cryptocurrencies before making investment decisions.
* **Stay informed:** Keep up-to-date with news and developments in the cryptocurrency market to make informed decisions.

**Disclaimer:** This analysis is for informational purposes only and should not be considered investment advice. Always conduct your own research and consult with a qualified financial advisor before making any investment decisions. 

    

Analysis report saved as BTC_analysis.md
